In [10]:
import firebase_admin
from firebase_admin import credentials
from firebase_admin import db
from google.colab import files
files.upload()

# Path to your service account key file
# Make sure you've uploaded this file to your Colab environment
SERVICE_ACCOUNT_KEY_FILE = 'serviceAccountKey.json'

# Your Firebase project's Realtime Database URL
# You can find this in your Firebase project console under Realtime Database -> Data tab
DATABASE_URL =

# Initialize the app with a service account, granting admin privileges
try:
    # Check if the app is already initialized to avoid re-initialization errors
    if not firebase_admin._apps:
        cred = credentials.Certificate(SERVICE_ACCOUNT_KEY_FILE)
        firebase_admin.initialize_app(cred, {
            'databaseURL': DATABASE_URL
        })
    print('Firebase Admin SDK initialized successfully.')
except FileNotFoundError:
    print(f"Error: Service account key file '{SERVICE_ACCOUNT_KEY_FILE}' not found. Please upload it to Colab.")
except Exception as e:
    print(f"Error initializing Firebase Admin SDK: {e}")

Error: Service account key file 'serviceAccountKey.json' not found. Please upload it to Colab.


In [11]:
import re


# Reference to the "words" collection/table in Realtime Database
words_ref = db.reference("words")


def clean_word(word):
    word = word.lower().strip()
    word = re.sub(r"[^a-zA-Z]", "", word)
    return word


def get_word_count(word):
    count = words_ref.child(word).get()

    if count is None:
        return 0

    return count


def save_word_count(word, count):
    words_ref.child(word).set(count)


def add_single_word():
    word = input("Enter a word: ")
    word = clean_word(word)

    if word == "":
        print("Invalid word.")
        return

    current_count = get_word_count(word)
    save_word_count(word, current_count + 1)

    print(f"Word '{word}' was added.")
    print(f"Current count: {current_count + 1}")


def add_text_for_analysis():
    text = input("Enter text: ")

    words = text.split()

    for word in words:
        word = clean_word(word)

        if word != "":
            current_count = get_word_count(word)
            save_word_count(word, current_count + 1)

    print("Text was analyzed successfully.")


def update_word_count():
    word = input("Enter word to update: ")
    word = clean_word(word)

    if word == "":
        print("Invalid word.")
        return

    try:
        new_count = int(input("Enter new count: "))
    except ValueError:
        print("Count must be a number.")
        return

    save_word_count(word, new_count)

    print(f"Word '{word}' was updated to {new_count}.")


def delete_word():
    word = input("Enter word to delete: ")
    word = clean_word(word)

    if word == "":
        print("Invalid word.")
        return

    words_ref.child(word).delete()

    print(f"Word '{word}' was deleted.")


def view_all_words():
    words = words_ref.get()

    if words is None:
        print("No words found.")
        return

    print("All words:")

    for word, count in words.items():
        print(f"{word}: {count}")


def menu():
    while True:
        print()
        print("Common Words Counter Menu:")
        print("1. Add single word")
        print("2. Add text for analysis")
        print("3. Update word count")
        print("4. Delete word")
        print("5. View all words")
        print("6. Exit")

        choice = input("Select an option (1-6): ")

        if choice == "1":
            add_single_word()

        elif choice == "2":
            add_text_for_analysis()

        elif choice == "3":
            update_word_count()

        elif choice == "4":
            delete_word()

        elif choice == "5":
            view_all_words()

        elif choice == "6":
            print("Goodbye!")
            break

        else:
            print("Invalid option. Please try again.")

ValueError: The default Firebase app does not exist. Make sure to initialize the SDK by calling initialize_app().